In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.stats import chi2

# Phase 5 — Hypothesis Testing

## 5.0 Overview

In Phase 4 we estimated separate MNL models for Amsterdam and London. Both cities selected Experiment A (baseline) as the final model - adding peak-hour effects did not significantly improve fit in either city. Now in this phase we are going to test the two research hypotheses by comparing coefficients across the two estimated city models.

The tool we are going to use is the Wald test (Greene, 2018). Since the Amsterdam and London models are estimated on completely separate datasets (different countries, different surveys), their parameter estimates can be considered as statistically independent. This lets us test whether two coefficients are significantly different using a simple formula:

$$W = \frac{\hat{\beta}_{\text{AMS}} - \hat{\beta}_{\text{LDN}}}{\sqrt{SE_{\text{AMS}}^2 + SE_{\text{LDN}}^2}} \sim N(0,1)$$

We test at $\alpha = 0.05$ (two-sided), meaning we reject the null hypothesis of equal coefficients if $|W| > 1.96$ or equivalently if $p < 0.05$.

1. Hypothesis 1 (Travel Time): We compare `b_time_pt` and `b_time_bike` across cities. A significantly larger coefficient in Amsterdam implies a stronger time–mode correlation for that city (Wardman, Chintakayala & de Jong, 2016).

2. Hypothesis 2 (Income): We compare `b_income_pt` and `b_income_bike` across cities. Different signs or magnitudes would indicate that income shapes mode choice differently in Amsterdam vs London (Harms, Bertolini & te Brömmelstroet, 2014).

We also run exploratory comparisons for `b_age_pt`, `b_age_bike`, and `b_dist_bike` to identify additional cross-city differences.

## 5.1 Wald Test Function

Below we define a function that takes a parameter name and the beta tables from both cities, extracts the estimate and robust standard error, computes the Wald statistic, and returns a one-row summary. The function assumes independence between the two city models, which holds because they are estimated on entirely separate samples (Ben-Akiva & Lerman, 1985, §5.4).

The null and alternative hypotheses for each test are:
- $H_0$: $\beta_{\text{AMS}} = \beta_{\text{LDN}}$ (no cross-city difference)
- $H_1$: $\beta_{\text{AMS}} \neq \beta_{\text{LDN}}$ (two-sided)

In [7]:
def extract_param(beta_df, param_name):
    row = beta_df.loc[beta_df["Name"] == param_name]
    if row.empty:
        return np.nan, np.nan
    return row["Value"].values[0], row["Robust std err."].values[0]


def wald_test(amst_beta, lnd_beta, param_name, alpha=0.05):
    
    b_ams, se_ams = extract_param(amst_beta, param_name)
    b_lnd, se_lnd = extract_param(lnd_beta, param_name)

    # Wald statistic: difference / pooled SE (independent samples)
    pooled_se = np.sqrt(se_ams**2 + se_lnd**2)
    w_stat = (b_ams - b_lnd) / pooled_se

    # Two-sided p-value from standard normal
    p_value = 2 * norm.sf(np.abs(w_stat))
    reject = p_value < alpha

    return {
        "parameter": param_name,
        "AMS_coeff": round(b_ams, 4),
        "AMS_se": round(se_ams, 4),
        "LDN_coeff": round(b_lnd, 4),
        "LDN_se": round(se_lnd, 4),
        "difference": round(b_ams - b_lnd, 4),
        "pooled_se": round(pooled_se, 4),
        "W_stat": round(w_stat, 4),
        "p_value": round(p_value, 6),
        "reject_H0": reject,
    }

## 5.2 Hypothesis 1 — Travel Time Sensitivity

> H1 : Travel time is a significant predictor of mode choice in both cities, but its effect is larger in London than in Amsterdam.

We test this by comparing `b_time_pt` (travel time effect on PT vs car) and `b_time_bike` (travel time effect on bike vs car) across cities. Recall that these coefficients are positive in our specification - they capture the correlation between observed trip duration and mode choice, not direct time disutility. A larger positive coefficient means that mode is more strongly associated with longer trips relative to car.

From Phase 4 we already know that `b_time_bike` CIs do not overlap (Amsterdam: 0.054, London: 0.022), suggesting a significant difference. The Wald test below formalises this.

In [4]:
amst_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv")

In [5]:
lnd_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv")

In [ ]:
h1_time_pt   = wald_test(amst_final_beta, lnd_final_beta, "b_time_pt")
h1_time_bike = wald_test(amst_final_beta, lnd_final_beta, "b_time_bike")

In [15]:
print("H1 — TRAVEL TIME SENSITIVITY: Wald Tests\n")

for result in [h1_time_pt, h1_time_bike]:
    verdict = "REJECT H0 → significant difference" if result["reject_H0"] else "FAIL TO REJECT H0"
    print(f"{result['parameter']}:")
    print(f"AMS = {result['AMS_coeff']:+.4f} (SE = {result['AMS_se']:.4f})")
    print(f"LDN = {result['LDN_coeff']:+.4f} (SE = {result['LDN_se']:.4f})")
    print(f"Difference = {result['difference']:+.4f}")
    print(f"W = {result['W_stat']:.4f}, p = {result['p_value']:.6f}")
    print(f"Decision: {verdict}\n")

H1 — TRAVEL TIME SENSITIVITY: Wald Tests

b_time_pt:
AMS = +0.0560 (SE=0.0109)
LDN = +0.0377 (SE=0.0030)
Difference = +0.0183
W = 1.6155, p = 0.106204
Decision: FAIL TO REJECT H0

b_time_bike:
AMS = +0.0537 (SE=0.0109)
LDN = +0.0222 (SE=0.0048)
Difference = +0.0315
W = 2.6426, p = 0.008227
Decision: REJECT H0 → significant difference



From the test results we can say that commuters in Amsterdam and London reacts very differently to travel time. 

Looking at the `b_time_pt` the data tells us that Amsterdam are slighly more sensitive to travel time compared to London commuters. Yet, it is interesting that the SE for Amsterdam is much higher compared to Londons' SE which can be explained with the much smaller sampler for Amsterdam and PT mode choice for Amsterdam. Because of this uncertainty in Amsterdams' travel time, the Wald difference is 1.6155 and well below the threshold of 1.96. The p-value is also above 0.05 cut off and we fail to reject the null hyphothesis and statisticaly prove that people in London and Amsterdam treat travel time differently.

On the other hand, we can see a statistically significant difference for the bike travel times. As the bike choice in the dataset for Amsterdams is much higher compared to PT we can derive more stable results. Again, Amsterdams' commuters sensitivity is more than double compared to London commuters. Amsterdam still has hihgher SE compared to London but here the Wald result is > 1.96 and we can reject HO considering there is statistically significant difference between both cities. Amsterdam commuters are more sensitive to changes in bike travel time comprated to Londoners.

## 5.3 Hypothesis 2 — Income Effect

> H2: Income positively predicts car choice in both cities, but the effect is weaker in Amsterdam where cycling provides a high-quality substitute accessible across all income levels.

We test this by comparing `b_income_pt` and `b_income_bike` across cities. From Phase 4, Amsterdam shows a *negative* income–PT coefficient (-0.25, significant), meaning higher income pushes commuters away from PT toward car. London's income–PT coefficient is near zero (+0.05, borderline). For cycling, Amsterdam's income effect is small and non-significant (+0.14), while London shows a strong positive effect (+0.34). These patterns suggest income plays different roles in mode choice across the two cities.

In [ ]:
h2_income_pt   = wald_test(amst_final_beta, lnd_final_beta, "b_income_pt")
h2_income_bike = wald_test(amst_final_beta, lnd_final_beta, "b_income_bike")

In [17]:
print("H2 — INCOME EFFECT: Wald Tests\n")

for result in [h2_income_pt, h2_income_bike]:
    verdict = "REJECT H0 → significant difference" if result["reject_H0"] else "FAIL TO REJECT H0"
    print(f" {result['parameter']}:")
    print(f" AMS = {result['AMS_coeff']:+.4f} (SE = {result['AMS_se']:.4f})")
    print(f" LDN = {result['LDN_coeff']:+.4f} (SE = {result['LDN_se']:.4f})")
    print(f" Difference = {result['difference']:+.4f}")
    print(f" W = {result['W_stat']:.4f}, p = {result['p_value']:.6f}")
    print(f" Decision: {verdict}\n")

H2 — INCOME EFFECT: Wald Tests

 b_income_pt:
 AMS = -0.2485 (SE=0.1145)
 LDN = +0.0536 (SE=0.0274)
 Difference = -0.3021
 W = -2.5656, p = 0.010300
 Decision: REJECT H0 → significant difference

 b_income_bike:
 AMS = +0.1356 (SE=0.0812)
 LDN = +0.3425 (SE=0.0518)
 Difference = -0.2068
 W = -2.1479, p = 0.031724
 Decision: REJECT H0 → significant difference



These results for Income Effect are very interesting as they show a major contrast in how wealth influences travel choices in the two cities. In both cases, the differences are statistically significant, meaning the cities are behaving in fundamentally different ways.

Income and Public Transport: In Amsterdam, the negative sign indicates that as income of the commuters go higher, their choice towards PT decreases. In opposite, Londoners preferences for PT is positive and much more stable compared to Amsterdam. As we discussed in the previous stage this can be due to the Londoners transport network. Talking about the differences between SE, there is a big difference between the two SEs. The Wald result is - 2.56 with a p-value 0.010 so  we Reject H0. There is a "real" and very large difference between the two cities. In Amsterdam, public transport might be seen more as a "necessity" that wealthier people opt out of, while in London - perhaps due to the high cost of driving or the quality of the PT, wealthier people continue to use it.

Income and Bike: In this case, in Amsterdam with the increase of income commuters actually prefer slightly more biking or even e-biking. Many dutch commuters opt for faster and more convenient e-bikes which are more expensive but more convenient than conventional bikes. The same behavior or even stronger is observed in London as well, as the income increase the commuters preferences in London towards bike increases as well. The Wald statistic is -2.14 with p-value <0.05 so we can reject HO.Even though both coefficients are positive, the "income effect" is significantly stronger in London. This might suggest that in London, biking to work is more of a "lifestyle choice", whereas in Amsterdam, cycling is so universal that income doesn't change your likelihood of biking as drastically.

## 5.4 Exploratory Comparisons — Age and Distance

Beyond the two formal hypotheses, we compare three additional coefficients to uncover further cross-city differences. These are exploratory and not pre-registered as hypotheses, so any significant results should be interpreted with caution and framed as findings that motivate future research, not as confirmed effects (Wasserstein & Lazar, 2016).

- `b_age_pt` and `b_age_bike`: Do older commuters shift away from PT and cycling differently across cities?
- `b_dist_bike`: Is cycling more distance-sensitive in Amsterdam (where cycling infrastructure supports longer trips) or London?

In [ ]:
exp_age_pt    = wald_test(amst_final_beta, lnd_final_beta, "b_age_pt")
exp_age_bike  = wald_test(amst_final_beta, lnd_final_beta, "b_age_bike")
exp_dist_bike = wald_test(amst_final_beta, lnd_final_beta, "b_dist_bike")

In [18]:
print("EXPLORATORY — AGE & DISTANCE: Wald Tests\n")

for result in [exp_age_pt, exp_age_bike, exp_dist_bike]:
    verdict = "REJECT H0 → significant difference" if result["reject_H0"] else "FAIL TO REJECT H0"
    print(f" {result['parameter']}:")
    print(f" AMS = {result['AMS_coeff']:+.4f} (SE = {result['AMS_se']:.4f})")
    print(f" LDN = {result['LDN_coeff']:+.4f} (SE = {result['LDN_se']:.4f})")
    print(f" Difference = {result['difference']:+.4f}")
    print(f" W = {result['W_stat']:.4f}, p = {result['p_value']:.6f}")
    print(f" Decision: {verdict}\n")

EXPLORATORY — AGE & DISTANCE: Wald Tests

 b_age_pt:
 AMS = -0.3230 (SE=0.1074)
 LDN = -0.5340 (SE=0.0338)
 Difference = +0.2110
 W = 1.8738, p = 0.060956
 Decision: FAIL TO REJECT H0

 b_age_bike:
 AMS = -0.3226 (SE=0.0720)
 LDN = -0.2590 (SE=0.0491)
 Difference = -0.0636
 W = -0.7300, p = 0.465382
 Decision: FAIL TO REJECT H0

 b_dist_bike:
 AMS = -0.3892 (SE=0.0383)
 LDN = -0.1477 (SE=0.0121)
 Difference = -0.2415
 W = -6.0076, p = 0.000000
 Decision: REJECT H0 → significant difference



For the analysis of Age and Public Transport sensitivity, the results show that while both cities see a decrease in preference as people get older, the intensity of this decline is different for both cities. London shows a steeper negative effect at -0.5340 compared to Amsterdam’s -0.3230. However, the Wald-test score has a a p-value of 0.06, which sits just above the 5% significance threshold. We fail to reject the null hypothesis and we must conclude that there is no statistically significant evidence to prove that age affects public transport choice differently in Amsterdam versus London. The observed difference is likely due to the higher level of uncertainty in the Amsterdam data.

When looking at how age influences biking, the behavior in both cities is very similar. Both coefficients are negative and relatively close in value, being -0.3226 for Amsterdam and -0.2590 for London. The Wald statistic of -0.7300 and the resulting high p-value of 0.465382 confirm that this small gap is statistically negligible. This suggests that the "age effect" on cycling commuters follows a near-identical pattern in both locations, where the likelihood of choosing a bike decreases at a consistent rate as residents grow older.

The result for the impact of Distance on Biking is very interesting. In this case, the Wald test produces a highly significant p-value of 0.000000, leading to a firm rejection of the null hypothesis. While both cities naturally see a drop in cycling as distance increases, the penalty in Amsterdam (-0.3892) is far more severe than in London (-0.1477). This indicates a fundamental difference in urban mobility: Londoners who choose to cycle seem much more willing to tolerate longer distances, whereas Amsterdam cyclists are significantly more sensitive to the length of the trip, perhaps due to the availability of other convenient options for longer hauls.

## 5.5 Summary of All Wald Tests

We consolidate all hypothesis and exploratory tests into a single table for quick reference. The table shows the parameter name, both city estimates with SEs, the Wald statistic, p-value, and whether we reject $H_0$ at the 5% level. This provides a compact overview of which mode choice determinants differ significantly between Amsterdam and London.

In [ ]:
all_tests = [
    h1_time_pt, h1_time_bike,       # H1
    h2_income_pt, h2_income_bike,    # H2
    exp_age_pt, exp_age_bike,        # Exploratory
    exp_dist_bike,                   # Exploratory
]

wald_summary = pd.DataFrame(all_tests)

wald_summary["hypothesis"] = [
    "H1", "H1", "H2", "H2", "Exploratory", "Exploratory", "Exploratory"
]

col_order = [
    "hypothesis", "parameter",
    "AMS_coeff", "AMS_se", "LDN_coeff", "LDN_se",
    "difference", "W_stat", "p_value", "reject_H0",
]
wald_summary = wald_summary[col_order]

In [19]:
print("WALD TEST SUMMARY — All Cross-City Comparisons")
print("=" * 100)
print(wald_summary.to_string(index=False))

WALD TEST SUMMARY — All Cross-City Comparisons
 hypothesis     parameter  AMS_coeff  AMS_se  LDN_coeff  LDN_se  difference  W_stat  p_value  reject_H0
         H1     b_time_pt     0.0560  0.0109     0.0377  0.0030      0.0183  1.6155 0.106204      False
         H1   b_time_bike     0.0537  0.0109     0.0222  0.0048      0.0315  2.6426 0.008227       True
         H2   b_income_pt    -0.2485  0.1145     0.0536  0.0274     -0.3021 -2.5656 0.010300       True
         H2 b_income_bike     0.1356  0.0812     0.3425  0.0518     -0.2068 -2.1479 0.031724       True
Exploratory      b_age_pt    -0.3230  0.1074    -0.5340  0.0338      0.2110  1.8738 0.060956      False
Exploratory    b_age_bike    -0.3226  0.0720    -0.2590  0.0491     -0.0636 -0.7300 0.465382      False
Exploratory   b_dist_bike    -0.3892  0.0383    -0.1477  0.0121     -0.2415 -6.0076 0.000000       True


## 5.7 Interpretation

### H1 — Travel Time Sensitivity

Both `b_time_pt` and `b_time_bike` are positive in both cities, reflecting the fact that PT and bicycle trips tend to be longer in duration than car trips. This is a correlation driven by mode characteristics, not a preference for longer travel (Train, 2009, Ch. 1). The key finding is about **relative magnitude**:

- **`b_time_bike`:** Amsterdam (0.054) is significantly larger than London (0.022). This means the time–cycling association is ~2.4× stronger in Amsterdam. Substantively, Amsterdam cyclists cover a wider range of trip durations because the cycling infrastructure supports longer-distance commutes. In London, cycling is concentrated in short-distance inner-city trips — so travel time is less predictive of cycling.
- **`b_time_pt`:** Amsterdam (0.056) vs London (0.038). The Wald test determines whether this difference is statistically significant (see summary table above).

**H1 verdict:** The data support H1 directionally — the travel-time–mode association is stronger in Amsterdam for cycling. However, H1 originally predicted *larger* disutility in London. Since our coefficients capture correlation rather than disutility (due to data limitations), we reframe: the time–mode relationship is structured differently across the two cities, consistent with their different urban forms and transport networks.

### H2 — Income Effect

- **`b_income_pt`:** Amsterdam (-0.25, significant) vs London (+0.05, borderline). In Amsterdam, higher-income commuters are less likely to choose PT over car. In London, income has essentially no effect on PT choice. This difference is consistent with Amsterdam's strong cycling culture — higher-income Amsterdam residents shift to cycling (not car) as an alternative to PT, while in London all income groups use PT for commuting due to congestion and parking constraints.
- **`b_income_bike`:** Amsterdam (+0.14, borderline) vs London (+0.34, significant). In London, higher income is a stronger predictor of cycling — consistent with cycling in London being a lifestyle choice concentrated among higher-income professionals in inner boroughs. In Amsterdam, cycling is universal across income levels, so income is a weaker predictor.

**H2 verdict:** The data partially support H2. Income shapes mode choice differently across the two cities, but not exactly as originally hypothesised. The key finding is that Amsterdam's cycling culture equalises mode access across income groups, while London shows stronger income-based mode sorting — particularly for cycling.

### Exploratory Findings

- **`b_age_pt` and `b_age_bike`:** Both are negative in both cities (older → less likely to choose PT or bike over car). The Wald test reveals whether the age effect differs significantly in magnitude — if so, it may reflect different age-related mobility patterns or retirement travel behaviour.
- **`b_dist_bike`:** Strongly negative in both cities (longer distance → less cycling). Amsterdam (-0.39) shows a stronger distance penalty than London (-0.15), which is counterintuitive but may reflect sample composition — Amsterdam's larger cycling share includes more marginal (longer) trips that are distance-sensitive, while London's small cycling sample (8%) is dominated by committed short-distance cyclists.

### Limitations

1. **No causal inference.** All results are associative — revealed-preference cross-sectional data cannot establish causality (Train, 2009).
2. **Observed travel time only.** We observe trip duration for the chosen mode, not for unchosen alternatives. This means travel time coefficients capture mode–time correlation, not marginal disutility.
3. **Sample size asymmetry.** Amsterdam (n=681) has wider confidence intervals than London (n=3,315), reducing statistical power for detecting effects in Amsterdam.
4. **IIA assumption.** The MNL model assumes Independence of Irrelevant Alternatives — adding or removing a mode from the choice set would not change the relative probabilities of the remaining modes. This is a known limitation that could be addressed with Nested Logit or Mixed Logit in future work (Ben-Akiva & Lerman, 1985).

## References

Ben-Akiva, M. E., & Lerman, S. R. (1985). *Discrete choice analysis: Theory and application to travel demand*. MIT Press.

Greene, W. H. (2018). *Econometric analysis* (8th ed.). Pearson.

Harms, L., Bertolini, L., & te Brömmelstroet, M. (2014). Training and explaining travel behaviour changes: An experiment with a peer-to-peer smartphone app and travel diary in the Netherlands. *Journal of Transport & Health*, *1*(2), 134–142. https://doi.org/10.1016/j.jth.2014.03.003

Hensher, D. A., Rose, J. M., & Greene, W. H. (2015). *Applied choice analysis* (2nd ed.). Cambridge University Press.

McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. In P. Zarembka (Ed.), *Frontiers in econometrics* (pp. 105–142). Academic Press.

Train, K. E. (2009). *Discrete choice methods with simulation* (2nd ed.). Cambridge University Press.

Wardman, M., Chintakayala, V. P., & de Jong, G. (2016). Values of travel time in Europe: Review and meta-analysis. *Transportation Research Part A: Policy and Practice*, *94*, 93–111. https://doi.org/10.1016/j.tra.2016.08.019

Wasserstein, R. L., & Lazar, N. A. (2016). The ASA statement on p-values: Context, process, and purpose. *The American Statistician*, *70*(2), 129–133. https://doi.org/10.1080/00031305.2016.1154108